# 图片存储与 App 回复审计

2026-10-17，新加坡时间。以下聚合值来自当次实际只读核查，不是模拟数据。原图、对话正文、安装标识和凭证不嵌入 notebook。运行后只重算既有快照，不表示重新查询服务器。

In [ ]:
import json
observed = {'capturedAt': '2026-10-17T01:17:08.846940+08:00', 'environment': 'demo', 'storedImages': 170, 'uniqueImageContents': 158, 'storedBytes': 40245926, 'emptyImages': 0, 'invalidImages': 0, 'completedCaptures': 148, 'completedWithSavedImage': 148, 'dashboardUniqueImages': 12, 'storedRowsNotDisplayed': 146, 'storedReplies': 95, 'dashboardReplies': 4, 'comparedReplies': 4, 'equalReplies': 4, 'comparedRepliesWithIOSReceipt': 3, 'iosRuns': 21, 'iosReplies': 21, 'repliesBeforeTraceWindow': 91, 'traceLimit': 20000, 'oldestTraceSingapore': '2026-10-16T22:56:23.055220+08:00'}
observed

## 实际使用的只读 SQL

两段分别在业务库与追踪库执行。SHA-256 按原始图片字节计算；回复按 run ID / trace ID 关联。

In [ ]:
image_sql = "SELECT id,mime,bytes FROM media;\nSELECT payload FROM events WHERE type='camera_completed';"
reply_sql = 'SELECT id,payload FROM runs;\nSELECT payload FROM records;'
print(image_sql, reply_sql, sep="\n")

In [ ]:
assert observed['storedImages'] == observed['uniqueImageContents'] + 12
assert observed['completedCaptures'] == observed['completedWithSavedImage']
assert observed['emptyImages'] == observed['invalidImages'] == 0
assert observed['comparedReplies'] == observed['equalReplies'] == 4
assert observed['storedReplies'] - observed['dashboardReplies'] == observed['repliesBeforeTraceWindow']
print('已保存图片完整性与四条回复一致性检查通过；Dashboard 的全量覆盖和手机逐字校验不成立。')

## 不能由聚合值推出的结论

- 不证明全部手机视频帧、失败上传或全部模型输入永久保存。
- 不证明所有历史任务的模型链路仍可追溯。
- 不证明客户端当前安装包与仓库完全相同或屏幕逐字显示一致。
- 两张真实图片的人工抽查不构成整体准确率。

详细发现与修复建议见 IMAGE-STORAGE-AND-APP-REPLY-AUDIT.md。